# Introductory Time Series with Python

Based on **Cowpertwait & Metcalfe — *Introductory Time Series with R* (2009)**, Chapters 1–6.

This notebook translates the R examples into idiomatic Python using `pandas`, `numpy`,
`matplotlib`, and `statsmodels`.

**Topics covered**
1. Exploring Time Series Data — trends, seasonal variation, decomposition
2. Correlation & the Correlogram — ACF, PACF
3. Forecasting Strategies — exponential smoothing, Holt-Winters
4. Basic Stochastic Models — white noise, random walk, AR processes
5. Time Series Regression — trend & seasonal models, GLS
6. Stationary Models — MA, ARMA, ARIMA, model selection

## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.regression.linear_model import GLSAR
import statsmodels.formula.api as smf

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True

---
## 1  Exploring Time Series Data
### 1.1  Loading the AirPassengers dataset

The classic monthly airline passenger dataset (Box & Jenkins, 1976) records
**thousands of international passengers per month** for Pan Am, 1949–1960.
It is the book's primary running example.

In [ ]:
# Classic AirPassengers dataset (144 monthly observations, Jan 1949 – Dec 1960)
ap_values = np.array([
    112, 118, 132, 129, 121, 135, 148, 148, 136, 119, 104, 118,
    115, 126, 141, 135, 125, 149, 170, 170, 158, 133, 114, 140,
    145, 150, 178, 163, 172, 178, 199, 199, 184, 162, 146, 166,
    171, 180, 193, 181, 183, 218, 230, 242, 209, 191, 172, 194,
    196, 196, 236, 235, 229, 243, 264, 272, 237, 211, 180, 201,
    204, 188, 235, 227, 234, 264, 302, 293, 259, 229, 203, 229,
    242, 233, 267, 269, 270, 315, 364, 347, 312, 274, 237, 278,
    284, 277, 317, 313, 318, 374, 413, 405, 355, 306, 271, 306,
    315, 301, 356, 348, 355, 422, 465, 467, 404, 347, 305, 336,
    340, 318, 362, 348, 363, 435, 491, 505, 404, 359, 310, 337,
    360, 342, 406, 396, 420, 472, 548, 559, 463, 407, 362, 405,
    417, 391, 419, 461, 472, 535, 622, 606, 508, 461, 390, 432,
], dtype=float)

ap_index = pd.date_range("1949-01", periods=144, freq="MS")
AP = pd.Series(ap_values, index=ap_index, name="Passengers (000s)")
print(AP.head(12))
print(f"\nLength: {len(AP)}, Start: {AP.index[0].date()}, End: {AP.index[-1].date()}")

### 1.2  Time plot — trend and seasonal variation

A time plot is the first step in any exploratory analysis.
Key features to look for:
- **Trend** — systematic upward or downward movement over time
- **Seasonal variation** — repeating pattern within each fixed period
- **Cycles** — longer-period oscillations (business cycles, climate)
- **Outliers / irregular events**

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Full series
axes[0].plot(AP.index, AP.values, color="steelblue")
axes[0].set_title("Monthly International Air Passengers, 1949–1960")
axes[0].set_ylabel("Passengers (000s)")

# Annual aggregation removes seasonal effect — cleaner trend view
annual = AP.resample("YE").mean()
axes[1].plot(annual.index, annual.values, marker="o", color="darkorange")
axes[1].set_title("Annual Mean Passengers (seasonal effect removed)")
axes[1].set_ylabel("Passengers (000s)")

plt.tight_layout()
plt.show()

# Seasonal box-plots: distribution within each calendar month
AP_df = AP.to_frame()
AP_df["month"] = AP.index.month
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(10, 4))
data_by_month = [AP_df[AP_df["month"] == m]["Passengers (000s)"].values for m in range(1, 13)]
ax.boxplot(data_by_month, labels=month_names)
ax.set_title("Seasonal Box-plots — Distribution by Calendar Month")
ax.set_ylabel("Passengers (000s)")
plt.tight_layout()
plt.show()

**Observations**
- Clear **upward trend**: passenger numbers roughly tripled from 1949 to 1960.
- Clear **seasonal variation**: peaks in summer (Jul–Aug), troughs in winter (Nov–Feb).
- The seasonal amplitude grows with the level → *multiplicative* seasonality.

### 1.3  Decomposition — additive vs multiplicative

A time series can be expressed as:

| Model | Formula |
|---|---|
| Additive | $x_t = T_t + S_t + R_t$ |
| Multiplicative | $x_t = T_t \times S_t \times R_t$ |

Use **additive** when seasonal swings are roughly constant; **multiplicative** when they
grow proportionally with the trend (as with AirPassengers).
`statsmodels.tsa.seasonal.seasonal_decompose` uses a centred moving-average trend.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

for ax, model in zip(axes, ["additive", "multiplicative"]):
    result = seasonal_decompose(AP, model=model, period=12)
    ax2 = ax.twinx()
    ax.plot(AP, label="Observed", color="steelblue", alpha=0.5)
    ax.plot(result.trend, label="Trend", color="darkorange", linewidth=2)
    ax2.plot(result.seasonal, label="Seasonal", color="green", linestyle="--", alpha=0.7)
    ax.set_title(f"{model.capitalize()} Decomposition — Trend + Seasonal overlay")
    ax.set_ylabel("Passengers (000s)")
    ax2.set_ylabel("Seasonal component", color="green")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

# Full decomposition panels for multiplicative model
result = seasonal_decompose(AP, model="multiplicative", period=12)
result.plot()
plt.suptitle("Multiplicative Decomposition: Trend / Seasonal / Residual", y=1.01)
plt.tight_layout()
plt.show()

---
## 2  Correlation & the Correlogram

### 2.1  Autocovariance and autocorrelation

For a stationary time series $\{x_t\}$, the **autocovariance function** (ACVF) at lag $k$ is:

$$\gamma_k = \text{Cov}(x_t, x_{t+k}) = E[(x_t - \mu)(x_{t+k} - \mu)]$$

The **autocorrelation function** (ACF) normalises by the variance:

$$\rho_k = \frac{\gamma_k}{\gamma_0}, \quad -1 \le \rho_k \le 1$$

Sample estimates use $\hat{\gamma}_k = \frac{1}{n}\sum_{t=1}^{n-k}(x_t - \bar{x})(x_{t+k} - \bar{x})$.

In [ ]:
# Manual ACF calculation for the first few lags
log_AP = np.log(AP)          # log-transform to stabilise variance
x = log_AP.values
n = len(x)
x_bar = x.mean()

print("Manual ACF vs statsmodels ACF (log-AirPassengers):")
print(f"{'Lag':>4}  {'Manual':>8}  {'statsmodels':>12}")
sm_acf = acf(x, nlags=6, fft=False)
for k in range(7):
    gamma_k = np.sum((x[:n-k] - x_bar) * (x[k:] - x_bar)) / n
    gamma_0 = np.sum((x - x_bar) ** 2) / n
    rho_k = gamma_k / gamma_0
    print(f"{k:>4}  {rho_k:>8.4f}  {sm_acf[k]:>12.4f}")

### 2.2  Correlogram (ACF) and Partial ACF

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# Original series ACF
plot_acf(AP, lags=36, ax=axes[0, 0], title="ACF — Original (levels)")
plot_pacf(AP, lags=36, ax=axes[0, 1], title="PACF — Original (levels)", method="ywm")

# Log-differenced (stationary): remove trend and stabilise variance
log_diff_AP = np.log(AP).diff().dropna()
plot_acf(log_diff_AP, lags=36, ax=axes[1, 0],
         title="ACF — Log-differenced (approx. stationary)")
plot_pacf(log_diff_AP, lags=36, ax=axes[1, 1],
          title="PACF — Log-differenced", method="ywm")

plt.tight_layout()
plt.show()

print("Key interpretations:")
print("  Original ACF: slowly decaying — non-stationary (trend present)")
print("  Log-diff ACF: spikes at lags 12, 24 → seasonal pattern of period 12")
print("  Log-diff PACF: spike at lag 1 → suggests AR(1) component")

**Reading the correlogram**

| ACF pattern | Suggests |
|---|---|
| Slow decay | Non-stationarity (needs differencing) |
| Cuts off at lag $q$ | MA($q$) process |
| Decays exponentially or sinusoidally | AR process |
| Spikes at seasonal lags (12, 24, …) | Seasonal component |

The **partial ACF** (PACF) shows the correlation at lag $k$ after removing
the effect of shorter lags. A PACF that cuts off at lag $p$ suggests AR($p$).

---
## 3  Forecasting Strategies — Exponential Smoothing

Exponential smoothing methods produce forecasts as **weighted averages of past
observations**, where weights decay exponentially with age.

### 3.1  Simple Exponential Smoothing (SES)

Appropriate for series **without trend or seasonality**.

$$\hat{x}_{t+1} = \alpha x_t + (1-\alpha)\hat{x}_t, \quad 0 < \alpha \le 1$$

Large $\alpha$ → more weight on recent observations (fast adaptation).
Small $\alpha$ → smoother, more inertia.

In [ ]:
# Demonstrate SES on the first-differenced series (remove trend first)
diff_AP = AP.diff().dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(diff_AP, label="First-differenced AP", color="steelblue", alpha=0.6)

for alpha in [0.1, 0.3, 0.8]:
    ses = SimpleExpSmoothing(diff_AP).fit(smoothing_level=alpha, optimized=False)
    ax.plot(ses.fittedvalues, label=f"SES α={alpha}", linewidth=1.5)

# Also show optimised alpha
ses_opt = SimpleExpSmoothing(diff_AP).fit(optimized=True)
ax.plot(ses_opt.fittedvalues, label=f"SES α={ses_opt.params['smoothing_level']:.3f} (opt)",
        linewidth=2, linestyle="--")

ax.set_title("Simple Exponential Smoothing on first-differenced AirPassengers")
ax.legend()
plt.tight_layout()
plt.show()

### 3.2  Holt's Linear Trend Method

Extends SES by adding a **trend component** $b_t$ — suitable for series with
trend but no seasonality.

$$l_t = \alpha x_t + (1-\alpha)(l_{t-1} + b_{t-1})$$
$$b_t = \beta(l_t - l_{t-1}) + (1-\beta)b_{t-1}$$
$$\hat{x}_{t+h} = l_t + h \cdot b_t$$

In [ ]:
# Holt's linear trend on annual aggregated series (no seasonality)
annual_ts = AP.resample("YE").mean()

holt = ExponentialSmoothing(annual_ts, trend="add", seasonal=None).fit()
print(f"Holt parameters: alpha={holt.params['smoothing_level']:.4f}, "
      f"beta={holt.params['smoothing_trend']:.4f}")

forecast = holt.forecast(5)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(annual_ts, marker="o", label="Observed (annual mean)", color="steelblue")
ax.plot(holt.fittedvalues, label="Fitted (Holt)", color="darkorange")
ax.plot(forecast, marker="s", linestyle="--", label="Forecast (5yr)", color="red")
ax.set_title("Holt's Linear Trend — Annual AirPassengers")
ax.set_ylabel("Passengers (000s)")
ax.legend()
plt.tight_layout()
plt.show()

### 3.3  Holt-Winters Seasonal Method

Extends Holt's method with a **seasonal component** $s_t$:

**Additive** (constant seasonal swings):
$$l_t = \alpha(x_t - s_{t-m}) + (1-\alpha)(l_{t-1}+b_{t-1})$$
$$s_t = \gamma(x_t - l_{t-1} - b_{t-1}) + (1-\gamma)s_{t-m}$$

**Multiplicative** (growing seasonal swings — appropriate for AirPassengers):
$$l_t = \alpha(x_t / s_{t-m}) + (1-\alpha)(l_{t-1}+b_{t-1})$$
$$s_t = \gamma(x_t / l_t) + (1-\gamma)s_{t-m}$$

In [ ]:
# Holt-Winters multiplicative on the full monthly series
hw = ExponentialSmoothing(
    AP, trend="add", seasonal="mul", seasonal_periods=12
).fit()

print("Holt-Winters parameters:")
print(f"  alpha (level)    = {hw.params['smoothing_level']:.4f}")
print(f"  beta  (trend)    = {hw.params['smoothing_trend']:.4f}")
print(f"  gamma (seasonal) = {hw.params['smoothing_seasonal']:.4f}")

forecast_hw = hw.forecast(24)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(AP, label="Observed", color="steelblue", alpha=0.7)
ax.plot(hw.fittedvalues, label="Holt-Winters fitted", color="darkorange", linewidth=1.5)
ax.plot(forecast_hw, label="Forecast (24 months)", color="red", linestyle="--", linewidth=2)
ax.axvline(AP.index[-1], color="gray", linestyle=":", linewidth=1)
ax.set_title("Holt-Winters Multiplicative — AirPassengers with 2-Year Forecast")
ax.set_ylabel("Passengers (000s)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4  Basic Stochastic Models

### 4.1  White Noise

A **white noise** process $\{w_t\}$ has:
- $E[w_t] = 0$ for all $t$
- $\text{Var}(w_t) = \sigma^2$ for all $t$
- $\text{Cov}(w_t, w_s) = 0$ for $t \neq s$

White noise is the building block for all ARIMA models.
The ACF should show **no significant autocorrelation** at any lag.

In [ ]:
rng = np.random.default_rng(42)
n = 300

white_noise = rng.normal(0, 1, n)
wn_series = pd.Series(white_noise, index=pd.date_range("2000-01", periods=n, freq="MS"))

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(wn_series, color="steelblue", linewidth=0.8)
axes[0].set_title("White Noise: Time Plot")
axes[0].axhline(0, color="red", linestyle="--")
plot_acf(wn_series, lags=30, ax=axes[1], title="White Noise: ACF (expect no significance)")
plt.tight_layout()
plt.show()

print(f"Mean: {white_noise.mean():.4f}  (expected 0)")
print(f"Std : {white_noise.std():.4f}  (expected 1)")

### 4.2  Random Walk and Stationarity

A **random walk** is $x_t = x_{t-1} + w_t$ — the cumulative sum of white noise.

- Non-stationary: mean and variance both grow with $t$
- First difference $\Delta x_t = x_t - x_{t-1} = w_t$ is stationary (white noise)
- Common model for financial prices and exchange rates

**Augmented Dickey-Fuller (ADF) test**
$H_0$: unit root present (non-stationary)
Reject $H_0$ at 5% if p-value < 0.05

In [ ]:
random_walk = np.cumsum(rng.normal(0, 1, n))
rw_series = pd.Series(random_walk, index=pd.date_range("2000-01", periods=n, freq="MS"))

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

axes[0, 0].plot(rw_series, color="steelblue")
axes[0, 0].set_title("Random Walk: Time Plot")

plot_acf(rw_series, lags=30, ax=axes[0, 1], title="Random Walk: ACF (slow decay)")

diff_rw = rw_series.diff().dropna()
axes[1, 0].plot(diff_rw, color="steelblue", linewidth=0.8)
axes[1, 0].set_title("First Difference of Random Walk (= white noise)")

plot_acf(diff_rw, lags=30, ax=axes[1, 1], title="First-Differenced: ACF")

plt.tight_layout()
plt.show()

# ADF test
for label, series in [("Random Walk (levels)", rw_series),
                       ("First Difference", diff_rw)]:
    adf_stat, p_val, _, _, crit, _ = adfuller(series)
    conclusion = "REJECT H0 (stationary)" if p_val < 0.05 else "FAIL to reject H0 (non-stationary)"
    print(f"{label}: ADF={adf_stat:.3f}, p={p_val:.4f} → {conclusion}")

### 4.3  Autoregressive Models — AR($p$)

$$x_t = \alpha_1 x_{t-1} + \alpha_2 x_{t-2} + \cdots + \alpha_p x_{t-p} + w_t$$

An AR($p$) process is **stationary** if all roots of $1 - \alpha_1 z - \cdots - \alpha_p z^p = 0$
lie **outside** the unit circle.

**ACF** of a stationary AR: decays exponentially (or oscillates and decays)
**PACF** of AR($p$): cuts off sharply after lag $p$ ← key diagnostic

In [ ]:
# Simulate AR(2): x_t = 0.6 x_{t-1} - 0.3 x_{t-2} + w_t
alpha = [0.6, -0.3]
n_sim = 500
x = np.zeros(n_sim)
w = rng.normal(0, 1, n_sim)
for t in range(2, n_sim):
    x[t] = alpha[0] * x[t-1] + alpha[1] * x[t-2] + w[t]

ar2_series = pd.Series(x, index=pd.date_range("2000-01", periods=n_sim, freq="MS"))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(ar2_series[:100], color="steelblue")
axes[0].set_title("AR(2) simulation (first 100 obs)")
plot_acf(ar2_series, lags=30, ax=axes[1], title="ACF — decays exponentially/sinusoidally")
plot_pacf(ar2_series, lags=30, ax=axes[2], title="PACF — cuts off at lag 2", method="ywm")
plt.tight_layout()
plt.show()

# Fit AR(2) with AutoReg
fit = AutoReg(ar2_series, lags=2).fit()
print(fit.summary().tables[1])

In [ ]:
# AR order selection using AIC
print("AR order selection by AIC:")
print(f"{'Order':>6}  {'AIC':>10}  {'BIC':>10}")
for p in range(1, 8):
    m = AutoReg(ar2_series, lags=p).fit()
    print(f"{p:>6}  {m.aic:>10.2f}  {m.bic:>10.2f}")
print("\nTrue order = 2; lowest AIC/BIC should select p=2")

---
## 5  Time Series Regression

### 5.1  Linear trend model

A deterministic trend can be modelled as:

$$x_t = \beta_0 + \beta_1 t + \varepsilon_t$$

where $\varepsilon_t$ may be autocorrelated — violating the OLS independence assumption.

In [ ]:
log_AP = np.log(AP)
df_reg = pd.DataFrame({
    "log_AP": log_AP.values,
    "t": np.arange(len(AP)) + 1,
    "month": AP.index.month,
}, index=AP.index)

# OLS: linear trend only
ols_trend = smf.ols("log_AP ~ t", data=df_reg).fit()
print("OLS linear trend fit:")
print(f"  Intercept: {ols_trend.params['Intercept']:.4f}")
print(f"  Trend coeff (beta_1): {ols_trend.params['t']:.6f}  (monthly growth rate)")
print(f"  Approx annual growth: {(np.exp(12 * ols_trend.params['t']) - 1)*100:.1f}%")

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(AP.index, log_AP, label="log(AP)", color="steelblue")
axes[0].plot(AP.index, ols_trend.fittedvalues, label="OLS trend", color="red", linewidth=2)
axes[0].set_title("Log AirPassengers with Linear Trend")
axes[0].legend()

resid = ols_trend.resid
axes[1].plot(AP.index, resid, color="steelblue")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Residuals from Linear Trend (seasonal pattern visible)")
plt.tight_layout()
plt.show()

### 5.2  Seasonal dummy variables

Adding monthly indicator variables absorbs the seasonal effect:

$$x_t = \beta_0 + \beta_1 t + \sum_{m=2}^{12} \delta_m D_{m,t} + \varepsilon_t$$

where $D_{m,t} = 1$ if observation $t$ falls in month $m$, else 0
(January is the baseline).

In [ ]:
# Add month dummies to the regression (January = baseline)
month_dummies = pd.get_dummies(df_reg["month"], prefix="m", drop_first=True, dtype=float)
df_full = pd.concat([df_reg, month_dummies], axis=1)

# Build formula with all month dummies
month_cols = " + ".join(month_dummies.columns)
formula = f"log_AP ~ t + {month_cols}"
ols_seasonal = smf.ols(formula, data=df_full).fit()

print(f"OLS with trend + seasonality:  R²={ols_seasonal.rsquared:.4f}")
print("\nMonth effects (relative to January):")
for col in month_dummies.columns:
    coeff = ols_seasonal.params[col]
    month_name = ["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][int(col.split("_")[1])]
    print(f"  {month_name}: {coeff:+.4f}  ({(np.exp(coeff)-1)*100:+.1f}% vs Jan)")

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(AP.index, log_AP, label="Observed", color="steelblue", alpha=0.6)
axes[0].plot(AP.index, ols_seasonal.fittedvalues, label="Fitted (trend+seasonal)", color="red")
axes[0].set_title("Log AirPassengers: OLS with Trend + Monthly Dummies")
axes[0].legend()

resid2 = ols_seasonal.resid
plot_acf(resid2, lags=30, ax=axes[1], title="ACF of Residuals — autocorrelation still present")
plt.tight_layout()
plt.show()

### 5.3  Generalised Least Squares (GLS) with AR(1) errors

When residuals are autocorrelated, OLS standard errors are **biased** (usually too small).
**GLS** accounts for the autocorrelation structure.

If $\varepsilon_t = \phi \varepsilon_{t-1} + w_t$ (AR(1) errors), `statsmodels.GLSAR`
iteratively estimates $\phi$ and re-fits until convergence.

In [ ]:
# Estimate AR(1) correlation from OLS residuals
rho_est = np.corrcoef(resid2[:-1], resid2[1:])[0, 1]
print(f"Estimated lag-1 autocorrelation in OLS residuals: {rho_est:.4f}")

# Build design matrix for GLS
X_cols = ["t"] + list(month_dummies.columns)
X = sm.add_constant(df_full[X_cols].values.astype(float))
y = log_AP.values

gls = GLSAR(y, X, rho=rho_est)
gls_result = gls.iterative_fit(maxiter=10)

print("\nOLS vs GLS standard errors for the trend coefficient:")
trend_idx = 1  # position of 't' in design matrix
print(f"  OLS SE: {ols_seasonal.bse['t']:.6f}")
print(f"  GLS SE: {gls_result.bse[trend_idx]:.6f}  (wider — corrects for autocorrelation)")
print(f"\nGLS trend coefficient: {gls_result.params[trend_idx]:.6f}")
print(f"  95% CI: ({gls_result.conf_int()[trend_idx, 0]:.6f}, "
      f"{gls_result.conf_int()[trend_idx, 1]:.6f})")

---
## 6  Stationary Models — MA, ARMA, ARIMA

### 6.1  Moving Average — MA($q$)

$$x_t = w_t + \beta_1 w_{t-1} + \cdots + \beta_q w_{t-q}$$

An MA($q$) process is **always stationary** and has:
- **ACF**: cuts off to zero after lag $q$ ← key diagnostic
- **PACF**: decays to zero (exponentially or with oscillations)

In [ ]:
# Simulate MA(3): x_t = w_t + 0.8 w_{t-1} + 0.6 w_{t-2} + 0.4 w_{t-3}
beta = [0.8, 0.6, 0.4]
n_sim = 1000
w_ma = rng.normal(0, 1, n_sim + 3)
x_ma = np.zeros(n_sim)
for t in range(n_sim):
    x_ma[t] = w_ma[t+3] + beta[0]*w_ma[t+2] + beta[1]*w_ma[t+1] + beta[2]*w_ma[t]

ma3_series = pd.Series(x_ma, index=pd.date_range("2000-01", periods=n_sim, freq="MS"))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(ma3_series[:100], color="steelblue", linewidth=0.8)
axes[0].set_title("MA(3) Simulation (first 100 obs)")
plot_acf(ma3_series, lags=20, ax=axes[1], title="ACF — cuts off after lag 3")
plot_pacf(ma3_series, lags=20, ax=axes[2], title="PACF — decays", method="ywm")
plt.tight_layout()
plt.show()

# Fit MA(3)
ma3_fit = ARIMA(ma3_series, order=(0, 0, 3)).fit()
print("MA(3) fitted parameters (true: [0.8, 0.6, 0.4]):")
print(ma3_fit.params[["ma.L1", "ma.L2", "ma.L3"]].round(4))

### 6.2  ARMA($p$, $q$) and ARIMA($p$, $d$, $q$)

The **ARMA($p,q$)** model combines AR and MA components:

$$x_t = \alpha_1 x_{t-1} + \cdots + \alpha_p x_{t-p} + w_t + \beta_1 w_{t-1} + \cdots + \beta_q w_{t-q}$$

**ARIMA($p,d,q$)** additionally differences the series $d$ times to achieve stationarity.
AirPassengers needs:
- $d=1$ (remove trend with first-difference)
- Log-transform first (stabilise variance → multiplicative → additive seasonality)

In [ ]:
# Fit ARIMA(2,1,1) to log-AirPassengers
log_AP_series = np.log(AP)
arima_fit = ARIMA(log_AP_series, order=(2, 1, 1)).fit()
print(arima_fit.summary().tables[1])

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(log_AP_series, label="Observed", color="steelblue")
axes[0].plot(arima_fit.fittedvalues, label="ARIMA(2,1,1) fitted", color="red", alpha=0.8)
axes[0].set_title("Log AirPassengers: ARIMA(2,1,1) fit")
axes[0].legend()

resid_arima = arima_fit.resid
plot_acf(resid_arima.dropna(), lags=30, ax=axes[1],
         title="ACF of ARIMA(2,1,1) residuals")
plt.tight_layout()
plt.show()

### 6.3  Model Selection — AIC / BIC grid search

**Akaike Information Criterion (AIC)** and **Bayesian IC (BIC)** penalise model
complexity:

$$\text{AIC} = -2\ln(L) + 2k, \qquad \text{BIC} = -2\ln(L) + k\ln(n)$$

where $k$ = number of parameters. **Lower is better.**
BIC penalises extra parameters more heavily and tends to select simpler models.

In [ ]:
print("ARIMA(p, 1, q) model selection on log-AirPassengers:")
print(f"{'(p,q)':>8}  {'AIC':>10}  {'BIC':>10}")

results = []
for p in range(0, 4):
    for q in range(0, 4):
        try:
            m = ARIMA(log_AP_series, order=(p, 1, q)).fit()
            results.append((p, q, m.aic, m.bic))
        except Exception:
            pass

results.sort(key=lambda r: r[2])  # sort by AIC
for p, q, aic, bic in results[:8]:
    print(f"({p},{q}):  {aic:>10.2f}  {bic:>10.2f}")

best_p, best_q = results[0][0], results[0][1]
print(f"\nBest model by AIC: ARIMA({best_p}, 1, {best_q})")

### 6.4  SARIMA — Seasonal ARIMA

For series with **seasonal autocorrelation**, the seasonal ARIMA model
ARIMA$(p,d,q)(P,D,Q)_m$ adds seasonal AR and MA terms at multiples of $m$:

$$\Phi(B^m)\phi(B)\nabla^D_m \nabla^d x_t = \Theta(B^m)\theta(B)w_t$$

For AirPassengers the standard benchmark is **SARIMA(0,1,1)(0,1,1)₁₂**
(Box-Jenkins "airline model").

In [ ]:
# Fit SARIMA(0,1,1)(0,1,1)_12 — the Box-Jenkins "airline model"
sarima = ARIMA(log_AP_series, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit()
print(sarima.summary().tables[1])

# Forecast 24 months ahead
forecast_result = sarima.get_forecast(24)
forecast_mean = np.exp(forecast_result.predicted_mean)
conf_int = np.exp(forecast_result.conf_int())

future_index = pd.date_range("1961-01", periods=24, freq="MS")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(AP, label="Observed", color="steelblue")
ax.plot(np.exp(sarima.fittedvalues), label="SARIMA fitted", color="darkorange", alpha=0.8)
ax.plot(future_index, forecast_mean.values, color="red", label="Forecast (24 months)",
        linestyle="--", linewidth=2)
ax.fill_between(future_index, conf_int.iloc[:, 0].values, conf_int.iloc[:, 1].values,
                alpha=0.2, color="red", label="95% CI")
ax.axvline(AP.index[-1], color="gray", linestyle=":", linewidth=1)
ax.set_title("SARIMA(0,1,1)(0,1,1)₁₂ — AirPassengers forecast with 95% CI")
ax.set_ylabel("Passengers (000s)")
ax.legend()
plt.tight_layout()
plt.show()

# Residual diagnostics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
resid_sarima = sarima.resid.dropna()
plot_acf(resid_sarima, lags=36, ax=axes[0], title="SARIMA residuals: ACF")
axes[1].hist(resid_sarima, bins=20, edgecolor="black", color="steelblue")
axes[1].set_title("SARIMA residuals: Histogram")
plt.tight_layout()
plt.show()

print(f"\nLjung-Box test on residuals (lag 24):")
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(resid_sarima, lags=[24], return_df=True)
print(lb.to_string())

---
## Summary

| Chapter | Key Concept | Python Tool |
|---|---|---|
| 1 | Trend, seasonality, decomposition | `seasonal_decompose`, `.resample()` |
| 2 | ACF / PACF correlogram | `plot_acf`, `plot_pacf` |
| 3 | Exponential smoothing, Holt-Winters | `ExponentialSmoothing` |
| 4 | White noise, random walk, AR($p$) | `adfuller`, `AutoReg` |
| 5 | Trend+seasonal OLS regression, GLS | `smf.ols`, `GLSAR` |
| 6 | MA, ARMA, ARIMA, SARIMA | `ARIMA`, AIC/BIC grid search |

**Model identification workflow (Box-Jenkins)**
1. Plot the series and examine for trend/seasonality
2. Transform if needed (log, difference) to achieve stationarity
3. Plot ACF and PACF to identify candidate $p, q$
4. Fit ARIMA, check AIC/BIC
5. Diagnose residuals (ACF should be white noise, Ljung-Box p > 0.05)
6. Forecast with confidence intervals